In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
!pip install gradio
import gradio as gr

In [3]:
!pip install transformers datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ecea2cb3811d9f4fe155a063b7f2213bd2bda9d65ffd0866f92e8ad18f8c6d42
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [4]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 52.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [6]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [ ]:
# drive.mount('/content/drive')

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [ ]:
# model_path = "/content/drive/MyDrive/headline-model"

# tokenizer = T5Tokenizer.from_pretrained(model_path)
# model = T5ForConditionalGeneration.from_pretrained(model_path).to(device)

# print("Model loaded successfully")

In [ ]:
print(dataset["train"][0])

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [ ]:
small_train = dataset["train"].shuffle(seed=42).select(range(50000))
small_val = dataset["validation"].shuffle(seed=42).select(range(2000))

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
model = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def preprocess(example):
    inputs = ["summarize: " + article for article in example["article"]]
    targets = [h.split("\n")[0] for h in example["highlights"]]

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=20, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_train = small_train.map(preprocess, batched=True)
tokenized_val = small_val.map(preprocess, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=True,
    load_best_model_at_end=True,
    save_total_limit=2
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.573787,1.332097
2,1.442311,1.309582
3,1.458904,1.307763


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=37500, training_loss=1.5217411767578124, metrics={'train_runtime': 4717.3125, 'train_samples_per_second': 31.798, 'train_steps_per_second': 7.949, 'total_flos': 2.03012702208e+16, 'train_loss': 1.5217411767578124, 'epoch': 3.0})

In [9]:
def generate_headline(text):
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            num_beams=8,
            min_length=6,
            max_length=32,
            length_penalty=0.8,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
def generate_multiple_headlines(text, num_return_sequences=3):
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=20,
            min_length=8,
            num_beams=8,
            num_return_sequences=num_return_sequences,
            no_repeat_ngram_size=2,
            early_stopping=False
        )

    headlines = [
        tokenizer.decode(output, skip_special_tokens=True)
        for output in outputs
    ]

    return headlines

In [11]:
def batch_generate_headlines(articles):
    results = []

    for article in articles:
        headline = generate_headline(article)
        results.append(headline)

    return results

In [12]:
def batch_multiple_interface(text, count):
    """
    Input multiple articles separated by ###
    and generate multiple headlines for each article
    """
    articles = [a.strip() for a in text.split("###") if a.strip()]
    count = int(count)

    output = ""

    for i, article in enumerate(articles, 1):
        output += f"ARTICLE {i}\n"

        headlines = generate_multiple_headlines(article, count)

        for j, headline in enumerate(headlines, 1):
            output += f"{j}. {headline}\n"

        output += "\n"

    return output

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

predictions = []
references = []

for i in range(100):  # you can increase later
    pred = generate_headline(dataset["test"][i]["article"])
    ref = dataset["test"][i]["highlights"].split("\n")[0]

    predictions.append(pred)
    references.append(ref)

results = rouge.compute(predictions=predictions, references=references)
print("ROUGE Scores:", results)

ROUGE Scores: {'rouge1': np.float64(0.2868094445360704), 'rouge2': np.float64(0.15068230695727491), 'rougeL': np.float64(0.26595922199074573), 'rougeLsum': np.float64(0.2668133770666045)}


In [ ]:
print("Generated:", generate_headline(dataset["test"][10]["article"]))
print("Actual:", dataset["test"][10]["highlights"])

Generated: Yahya Rashid, 19, was arrested at Luton airport on Tuesday.
Actual: London's Metropolitan Police say the man was arrested at Luton airport after landing on a flight from Istanbul .
He's been charged with terror offenses allegedly committed since the start of November .


In [ ]:
article = dataset["test"][0]["article"]

headlines = generate_multiple_headlines(article, 3)

for i, headline in enumerate(headlines, 1):
    print(f"{i}. {headline}")

1. The Palestinian Authority officially becomes the 123rd member of the International Criminal Court.
2. The Palestinian Authority becomes the 123rd member of the International Criminal Court.
3. The Palestinian Authority officially became the 123rd member of the International Criminal Court.


In [ ]:
articles = [
    dataset["test"][0]["article"],
    dataset["test"][1]["article"],
    dataset["test"][2]["article"]
]

headlines = batch_generate_headlines(articles)

for i, headline in enumerate(headlines, 1):
    print(f"Article {i}: {headline}")

Article 1: The Palestinian Authority officially becomes the 123rd member of the International Criminal Court.
Article 2: A stray pooch in Washington State has used up at least three of her own after
Article 3: Mohammad Javad Zarif has been U.S. Secretary of State John Kerry'


In [ ]:
!ls /content/drive/MyDrive

headline-model


In [ ]:
output_dir = "/content/drive/MyDrive/headline-model"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to /content/drive/MyDrive/headline-model


In [ ]:
import os
import shutil
from google.colab import files

# -----------------------------
# 1. Save model + tokenizer
# -----------------------------
save_directory = "news_model"

# Clean old directory if exists (avoids corrupted saves)
if os.path.exists(save_directory):
    shutil.rmtree(save_directory)

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model saved at: {save_directory}")

# -----------------------------
# 2. Create ZIP file
# -----------------------------
zip_path = shutil.make_archive(save_directory, 'zip', save_directory)

print(f"ZIP created at: {zip_path}")

# -----------------------------
# 3. Download ZIP
# -----------------------------
print("Downloading model...")
files.download(zip_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: news_model
ZIP created at: /content/news_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import HfApi, upload_folder

repo_name = "news-headline-generator"

api = HfApi()

api.create_repo(
    repo_id=f"himanshu0913/{repo_name}",
    repo_type="model",
    exist_ok=True
)

upload_folder(
    folder_path="news_model",
    repo_id=f"himanshu0913/{repo_name}",
    repo_type="model"
)

print("Model uploaded successfully!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s_model/model.safetensors:   0%|          |  552kB /  242MB            

Model uploaded successfully!


In [13]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

repo_id = "himanshu0913/news-headline-generator"

tokenizer = T5Tokenizer.from_pretrained(repo_id)
model = T5ForConditionalGeneration.from_pretrained(repo_id).to(device)

print("Loaded successfully from Hugging Face")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Loaded successfully from Hugging Face


In [14]:
demo = gr.TabbedInterface(
    [
        gr.Interface(
            fn=generate_headline,
            inputs=gr.Textbox(lines=12, label="Article"),
            outputs=gr.Textbox(
                lines=6,
                label="Generated Headline"
            ),
            title="Single Article → Single Headline"
        ),

        gr.Interface(
            fn=generate_multiple_headlines,
            inputs=[
                gr.Textbox(lines=12, label="Article"),
                gr.Slider(1, 5, value=3, step=1, label="Number of Headlines")
            ],
            outputs=gr.Textbox(
                lines=10,
                label="Generated Headlines"
            ),
            title="Single Article → Multiple Headlines"
        ),

        gr.Interface(
            fn=lambda text: batch_generate_headlines(
                [a.strip() for a in text.split("###") if a.strip()]
            ),
            inputs=gr.Textbox(
                lines=15,
                label="Articles separated by ###"
            ),
            outputs=gr.Textbox(
                lines=12,
                label="Generated Headlines"
            ),
            title="Multiple Articles → Single Headlines"
        ),

        gr.Interface(
            fn=batch_multiple_interface,
            inputs=[
                gr.Textbox(
                    lines=15,
                    label="Articles separated by ###"
                ),
                gr.Slider(
                    1, 5,
                    value=3,
                    step=1,
                    label="Headlines per article"
                )
            ],
            outputs=gr.Textbox(
                lines=20,
                label="Results"
            ),
            title="Multiple Articles → Multiple Headlines"
        )
    ],
    tab_names=[
        "1 Article → 1 Headline",
        "1 Article → Multiple Headlines",
        "Multiple Articles",
        "Multiple Articles + Multiple Headlines"
    ]
)

In [15]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://33de924caa1a2f36a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
article = """
Global technology companies are increasing investments in artificial intelligence as competition in the sector continues to intensify. On Tuesday, a leading multinational software company announced a new suite of AI-powered productivity tools designed to automate repetitive office tasks, generate reports, and assist developers with code generation.

The company stated that the new tools will be integrated directly into its existing cloud ecosystem, allowing businesses to deploy AI assistants across email, documentation platforms, customer service workflows, and internal analytics dashboards.

Industry analysts believe this move is part of a larger trend in which enterprises are shifting from experimental AI pilots to full-scale production adoption. According to recent market research, global spending on generative AI infrastructure is expected to exceed $200 billion by 2028.

However, the rapid expansion of AI technologies has also raised concerns among regulators and privacy advocates. Critics argue that increased automation may affect employment in administrative and support roles, while also introducing new risks related to data security, misinformation, and algorithmic bias.

Government agencies in multiple countries are currently drafting regulatory frameworks aimed at balancing innovation with accountability. These proposed regulations include transparency requirements for AI-generated content, mandatory security audits, and stricter guidelines on training data usage.

Despite these concerns, investors responded positively to the announcement, with the company’s stock price rising nearly 6% during afternoon trading. Executives emphasized that responsible AI deployment remains central to their long-term strategy and pledged continued collaboration with policymakers, researchers, and enterprise customers.
"""

inputs = tokenizer(
    "summarize: " + article,
    return_tensors="pt"
).to(device)

output = model.generate(inputs["input_ids"], max_length=20)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Global software company announced new suite of AI-powered productivity tools.


In [ ]:
article = """
Global technology companies are increasing investments in artificial intelligence as competition in the sector continues to intensify. On Tuesday, a leading multinational software company announced a new suite of AI-powered productivity tools designed to automate repetitive office tasks, generate reports, and assist developers with code generation.

The company stated that the new tools will be integrated directly into its existing cloud ecosystem, allowing businesses to deploy AI assistants across email, documentation platforms, customer service workflows, and internal analytics dashboards.

Industry analysts believe this move is part of a larger trend in which enterprises are shifting from experimental AI pilots to full-scale production adoption. According to recent market research, global spending on generative AI infrastructure is expected to exceed $200 billion by 2028.

However, the rapid expansion of AI technologies has also raised concerns among regulators and privacy advocates. Critics argue that increased automation may affect employment in administrative and support roles, while also introducing new risks related to data security, misinformation, and algorithmic bias.

Government agencies in multiple countries are currently drafting regulatory frameworks aimed at balancing innovation with accountability. These proposed regulations include transparency requirements for AI-generated content, mandatory security audits, and stricter guidelines on training data usage.

Despite these concerns, investors responded positively to the announcement, with the company’s stock price rising nearly 6% during afternoon trading. Executives emphasized that responsible AI deployment remains central to their long-term strategy and pledged continued collaboration with policymakers, researchers, and enterprise customers.
"""

print(generate_headline(article))
print(generate_multiple_headlines(article))

Leading multinational software company announced a new suite of AI-powered productivity tools.
['Leading multinational software company announced a new suite of AI-powered productivity tools.', 'The company announced a new suite of AI-powered productivity tools.', 'Leading multinational software company announced a suite of AI-powered productivity tools.']


In [ ]:
articles = [
    dataset["test"][0]["article"],
    dataset["test"][1]["article"],
    dataset["test"][2]["article"],
    dataset["test"][3]["article"],
    dataset["test"][4]["article"],
    dataset["test"][5]["article"],
    dataset["test"][6]["article"],
    dataset["test"][7]["article"],
    dataset["test"][8]["article"],
    dataset["test"][9]["article"]
]

headlines = batch_generate_headlines(articles)

for i, headline in enumerate(headlines, 1):
    print(f"Article {i}: {headline}")

Article 1: The Palestinian Authority officially becomes the 123rd member of the International Criminal Court.
Article 2: A stray pooch in Washington State has used up at least three of her own after
Article 3: Mohammad Javad Zarif has been U.S. Secretary of State John Kerry'
Article 4: Five Americans were monitored for three weeks after being exposed to Ebola in West Africa.
Article 5: The student was identified during an investigation by campus police and the office of student affairs.
Article 6: Trey Moses and Ellie Meredith have Down syndrome.
Article 7: Amnesty International says governments are using the threat of terrorism to advance executions 
Article 8: Andrew Getty, 47, had "several health issues," detective says.
Article 9: Maysak is classified as a tropical storm, according to the Philippine national weather service 
Article 10: Bob Barker hosted the TV game show for 35 years before stepping down in 2007.


In [ ]:
import time

times = []

for i in range(10):
    article = dataset["test"][i]["article"]

    start = time.time()
    _ = generate_headline(article)
    end = time.time()

    times.append(end - start)

avg_time = sum(times) / len(times)

print("Average inference time:", avg_time, "seconds")

Average inference time: 0.386678147315979 seconds


In [ ]:
for i in range(5):
    print("\nEXAMPLE", i+1)
    print("Generated:", generate_headline(dataset["test"][i]["article"]))
    print("Actual:", dataset["test"][i]["highlights"].split("\n")[0])


EXAMPLE 1
Generated: The Palestinian Authority officially becomes the 123rd member of the International Criminal Court.
Actual: Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .

EXAMPLE 2
Generated: A stray pooch in Washington State has used up at least three of her own after
Actual: Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field .

EXAMPLE 3
Generated: Mohammad Javad Zarif has been U.S. Secretary of State John Kerry'
Actual: Mohammad Javad Zarif has spent more time with John Kerry than any other foreign minister .

EXAMPLE 4
Generated: Five Americans were monitored for three weeks at Omaha, Nebraska, hospital.
Actual: 17 Americans were exposed to the Ebola virus while in Sierra Leone in March .

EXAMPLE 5
Generated: The student was identified during an investigation by campus police and the office of student affairs.
Actual: Student is no longer on Duke University campu